Lab 9 Naive Bayes Catergorical

Robert Armenta, Thai Vang

Dataset URL link: https://www.kaggle.com/datasets/uciml/adult-census-income

In [ ]:
# Step 1: Import tools
import pandas as pd                             # For working with data tables
from sklearn.naive_bayes import CategoricalNB   # Naive Bayes model for categorical data
from sklearn.preprocessing import LabelEncoder  # Converts words into numbers
from sklearn.model_selection import train_test_split  # Splits data into train/test sets
from sklearn.metrics import accuracy_score, classification_report  # Model evaluation tools



In [ ]:
#load the dataset

df = pd.read_csv("adult.csv")   # Load CSV file into a DataFrame

print("=== STEP 2: Original Dataset ===")
print(df.head())               # Show first 5 rows
print("Shape:", df.shape)      # Show rows and columns
print()



In [ ]:
#Select only categorical columns
# Remove numeric columns to match the tennis example style

df = df[
    [
        "workclass",
        "education",
        "marital.status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "native.country",
        "income"
    ]
].copy()



In [ ]:
 #Clean the data
# - Remove extra spaces
# - Replace missing values (?) with "Unknown"
# ============================================================

for col in df.columns:
    df[col] = df[col].astype(str).str.strip()   # Remove spaces like " Private"

df = df.replace("?", "Unknown")  # Replace missing values

print("=== STEP 4: Cleaned Dataset ===")
print(df.head())
print()

# Count how many people are in each income group
print(f"<=50K: {sum(df['income'] == '<=50K')}")
print(f">50K : {sum(df['income'] == '>50K')}")
print()



In [ ]:
#Convert categories (words) into numbers
# Each column gets its own LabelEncoder
# ============================================================

encoders = {}   # Dictionary to store encoders for each column

for col in df.columns:
    encoder = LabelEncoder()                      # Create encoder
    df[col + "_num"] = encoder.fit_transform(df[col])  # Convert words → numbers
    encoders[col] = encoder                       # Save encoder for later use

print("=== STEP 5: Encoded Dataset ===")
print(df.head())
print()


In [ ]:
 #Split data into inputs (X) and output (y)
# X = features used to make prediction
# y = correct answer (income)
# ============================================================

feature_columns = [
    "workclass_num",
    "education_num",
    "marital.status_num",
    "occupation_num",
    "relationship_num",
    "race_num",
    "sex_num",
    "native.country_num"
]

X = df[feature_columns]   # Input features
y = df["income_num"]      # Output label

print("=== STEP 6: Features and Label ===")
print("X shape:", X.shape)
print("y shape:", y.shape)
print()



In [ ]:
#Split into training and testing data
# Training data = used to teach the model
# Testing data  = used to evaluate performance
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,   # 20% test data
    random_state=42   # Keeps results consistent
)

print("===  Train/Test Split ===")
print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))
print()


In [ ]:
#Train the model
# ============================================================

model = CategoricalNB()     # Create model
model.fit(X_train, y_train) # Train model using training data

print("=== STEP 8: Model Trained ===")
print()


In [ ]:
#Evaluate the model
# ============================================================

y_pred = model.predict(X_test)   # Predict on test data

accuracy = accuracy_score(y_test, y_pred)   # Calculate accuracy

print("=== STEP 9: Model Evaluation ===")
print(f"Accuracy: {accuracy * 100:.2f}%")
print()

# Show detailed performance (precision, recall, etc.)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=encoders["income"].classes_))
print()



In [ ]:
#Predict for new people
# IMPORTANT: We must use transform(), not fit_transform()
# ============================================================

new_people = [
    {
        "workclass": "Private",
        "education": "Bachelors",
        "marital.status": "Never-married",
        "occupation": "Adm-clerical",
        "relationship": "Not-in-family",
        "race": "White",
        "sex": "Male",
        "native.country": "United-States"
    },
    {
        "workclass": "Self-emp-not-inc",
        "education": "Masters",
        "marital.status": "Married-civ-spouse",
        "occupation": "Exec-managerial",
        "relationship": "Husband",
        "race": "White",
        "sex": "Male",
        "native.country": "United-States"
    },
    {
        "workclass": "Private",
        "education": "HS-grad",
        "marital.status": "Divorced",
        "occupation": "Sales",
        "relationship": "Own-child",
        "race": "Black",
        "sex": "Female",
        "native.country": "United-States"
    }
]


In [ ]:
#"=== STEP 10: Predictions ===")
print()

for person in new_people:
    # Clean inputs (remove extra spaces if any)
    for key in person:
        person[key] = str(person[key]).strip()

    # Convert each category into its numeric value
    single_input = pd.DataFrame([{
    "workclass_num": encoders["workclass"].transform([person["workclass"]])[0],
    "education_num": encoders["education"].transform([person["education"]])[0],
    "marital.status_num": encoders["marital.status"].transform([person["marital.status"]])[0],
    "occupation_num": encoders["occupation"].transform([person["occupation"]])[0],
    "relationship_num": encoders["relationship"].transform([person["relationship"]])[0],
    "race_num": encoders["race"].transform([person["race"]])[0],
    "sex_num": encoders["sex"].transform([person["sex"]])[0],
    "native.country_num": encoders["native.country"].transform([person["native.country"]])[0]
}])

    # Predict income
    prediction = model.predict(single_input)[0]

    # Get probability of each class
    probabilities = model.predict_proba(single_input)[0]

    # Convert numeric prediction back to label (<=50K or >50K)
    result = encoders["income"].inverse_transform([prediction])[0]

    print("Person:")
    print(person)
    print(f"  --> Predicted income: {result}")
    print(f"  --> Probability <=50K: {probabilities[0] * 100:.1f}%")
    print(f"  --> Probability >50K : {probabilities[1] * 100:.1f}%")
    print()

**Lab Report**

We built a classification model using the Adult dataset to predict whether a person earns more than 50K per year based on categorical factors such as workclass, education, marital status, occupation, relationship, race, sex, and country. The data is cleaned by removing extra spaces and replacing missing values, Then all categories wree converted into numeric form so that the model could process them. A categorical naive bayes was tranied and tested on separate portions of the data to evaluate performance. The model showed a strong predictive ability, correctly identifying income levels with high confidence in most cases. For example, individuals with higher education and managerial roles were predicted to earn above 50k with very high probability, while others with lower education or entry level roles were predicted below 50k. Overall, the model demostrates that catergorical demographics and employment features can be used effectively to estimate income levels, though results should be interpreted as patterns rather than exact outcomes.